# 🚀 iOS App Builder
### Powered by NVIDIA NIM

## Prerequisites

Get your NVIDIA NIM API Key: [Click here](https://build.nvidia.com/nvidia/llama-3.1-nemotron-70b-instruct?signin=true&api_key=true)

In [ ]:
import getpass, os, subprocess, re, json
from IPython.display import display, Markdown

key = getpass.getpass("Enter NVIDIA API key: ")
assert key.startswith("nvapi-"), "Invalid key"
os.environ["NVIDIA_API_KEY"] = key
print("✅ API Key configured")

In [ ]:
from openai import OpenAI
import time

client = OpenAI(base_url="https://integrate.api.nvidia.com/v1", api_key=os.environ["NVIDIA_API_KEY"])
MODEL = "nvidia/llama-3.1-nemotron-70b-instruct"  # More reliable model

def ask(prompt, system="You are an expert iOS/SwiftUI developer.", retries=3):
    for i in range(retries):
        try:
            r = client.chat.completions.create(model=MODEL, messages=[
                {"role": "system", "content": system},
                {"role": "user", "content": prompt}
            ], temperature=0.3, max_tokens=4096)
            return r.choices[0].message.content
        except Exception as e:
            if i < retries - 1:
                print(f"⚠️ Retry {i+1}/{retries}: {e}")
                time.sleep(2)
            else:
                raise

def bash(cmd, timeout=120):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=timeout)
    return (r.stdout + r.stderr).strip()

def write_file(name, content):
    with open(os.path.join(PROJECT_PATH, name), 'w') as f: f.write(content)
    print(f"✅ {name}")

def extract_code(text):
    m = re.search(r'```(?:swift)?\n([\s\S]*?)```', text)
    return m.group(1).strip() if m else text.strip()

print(f"✅ {MODEL} ready")

---
# Phase 1: Describe Your App

In [ ]:
APP_DESC = input("📱 Describe your app idea: ")

In [ ]:
suggestion = ask(f"Suggest a short, catchy app name (one word, PascalCase) for: {APP_DESC}. Reply with ONLY the name.")
suggested_name = re.sub(r'[^a-zA-Z0-9]', '', suggestion.strip().split()[0])
print(f"🤖 Suggested: {suggested_name}")

In [ ]:
APP_NAME = input(f"📱 App Name [{suggested_name}]: ") or suggested_name
PROJECT_PATH = f"/Users/home/Documents/iOS/{APP_NAME}/{APP_NAME}"
XCODEPROJ = f"/Users/home/Documents/iOS/{APP_NAME}/{APP_NAME}.xcodeproj"
print(f"✅ {APP_NAME}")

In [ ]:
if not os.path.exists(PROJECT_PATH):
    print(f"⚠️ Create Xcode project first:")
    print(f"1. Xcode → File → New → Project → App")
    print(f"2. Product Name: {APP_NAME}")
    print(f"3. Save to: /Users/home/Documents/iOS/")
    input("Press Enter when done...")
print(f"✅ Project ready")

---
# Phase 2: Data Model

In [ ]:
model_prompt = f"""For an iOS app: {APP_DESC}

Design a Swift data model. Reply with ONLY a JSON object:
{{"name": "ModelName", "properties": [["propName", "Type"], ...]}}

Types: String, Int, Bool, Date, Double. Keep it simple (3-5 properties)."""

model_json = ask(model_prompt)
match = re.search(r'\{[^{}]*\}', model_json, re.DOTALL)
if match:
    model_data = json.loads(match.group())
    MODEL_NAME = model_data["name"]
    PROPS = [tuple(p) for p in model_data["properties"]]
else:
    MODEL_NAME, PROPS = "Item", [("name", "String")]

print(f"🤖 Model: {MODEL_NAME}")
for n, t in PROPS: print(f"   • {n}: {t}")

In [ ]:
accept = input("Accept model? [y]: ") or "y"
if accept.lower() == "n":
    MODEL_NAME = input("Model name: ") or MODEL_NAME
    print("Enter properties (name: Type), empty when done:")
    PROPS = []
    while True:
        p = input()
        if not p: break
        if ":" in p:
            n, t = [x.strip() for x in p.split(":", 1)]
            PROPS.append((n, t))
    if not PROPS: PROPS = [("name", "String")]

DISPLAY_PROP = next((n for n,t in PROPS if t=="String"), PROPS[0][0])
print(f"✅ {MODEL_NAME}")

---
# Phase 3: Features

In [ ]:
features_prompt = f"""For app: {APP_DESC} with model {MODEL_NAME}
Recommend features. Reply ONLY with JSON:
{{"nav": "list" or "tabs", "tabs": ["Tab1", "Tab2"] if tabs, "edit": true/false, "search": true/false}}"""

feat_json = ask(features_prompt)
match = re.search(r'\{[^{}]*\}', feat_json, re.DOTALL)
if match:
    feat = json.loads(match.group())
    NAV_STYLE = "tabs" if feat.get("nav") == "tabs" else "list"
    TABS = feat.get("tabs", [])
    EDIT = feat.get("edit", True)
    SEARCH = feat.get("search", False)
else:
    NAV_STYLE, TABS, EDIT, SEARCH = "list", [], True, False

print(f"🤖 Nav: {NAV_STYLE}" + (f" ({', '.join(TABS)})" if TABS else ""))
print(f"   Edit: {EDIT}, Search: {SEARCH}")

In [ ]:
accept = input("Accept features? [y]: ") or "y"
if accept.lower() == "n":
    NAV_STYLE = input("Nav style (list/tabs): ") or NAV_STYLE
    if NAV_STYLE == "tabs":
        TABS = input("Tab names (comma-separated): ").split(",")
    EDIT = input("Edit feature? (y/n): ").lower() != "n"
    SEARCH = input("Search feature? (y/n): ").lower() == "y"
print(f"✅ Features confirmed")

---
# Phase 4: Generate Swift Files

In [ ]:
# Generate Model
props_str = ', '.join(f'{n}: {t}' for n,t in PROPS)
model_code = ask(f"""Write a Swift struct for {MODEL_NAME} with properties: {props_str}
Make it Identifiable with UUID. Include a static sample property.
Reply with ONLY the Swift code, no explanation.""")
model_code = extract_code(model_code)
write_file(f"{MODEL_NAME}.swift", model_code)
display(Markdown(f"```swift\n{model_code}\n```"))

In [ ]:
# Generate ViewModel
vm_code = ask(f"""Write a SwiftUI ViewModel class for {MODEL_NAME} with @Published items array.
Include add, delete, update methods. Use @Observable macro (iOS 17+).
Reply with ONLY the Swift code.""")
vm_code = extract_code(vm_code)
write_file(f"{MODEL_NAME}ViewModel.swift", vm_code)
display(Markdown(f"```swift\n{vm_code}\n```"))

In [ ]:
# Generate Row View
row_code = ask(f"""Write a SwiftUI row view for {MODEL_NAME} showing {DISPLAY_PROP}.
Simple, clean design. Reply with ONLY the Swift code.""")
row_code = extract_code(row_code)
write_file(f"{MODEL_NAME}Row.swift", row_code)
display(Markdown(f"```swift\n{row_code}\n```"))

In [ ]:
# Generate Detail View
detail_code = ask(f"""Write a SwiftUI detail view for {MODEL_NAME}.
Show all properties: {props_str}. Clean layout.
Reply with ONLY the Swift code.""")
detail_code = extract_code(detail_code)
write_file(f"{MODEL_NAME}DetailView.swift", detail_code)
display(Markdown(f"```swift\n{detail_code}\n```"))

In [ ]:
# Generate Edit View (if enabled)
if EDIT:
    edit_code = ask(f"""Write a SwiftUI edit/add form for {MODEL_NAME}.
Properties: {props_str}. Use @Binding or @Environment for dismiss.
Reply with ONLY the Swift code.""")
    edit_code = extract_code(edit_code)
    write_file(f"{MODEL_NAME}EditView.swift", edit_code)
    display(Markdown(f"```swift\n{edit_code}\n```"))
else:
    print("⏭️ Edit view skipped")

In [ ]:
# Generate List View
search_str = "Include search bar filtering by " + DISPLAY_PROP if SEARCH else "No search needed"
list_code = ask(f"""Write a SwiftUI list view for {MODEL_NAME}.
Use NavigationStack, show {MODEL_NAME}Row for each item.
{search_str}. Add button to create new items.
Reply with ONLY the Swift code.""")
list_code = extract_code(list_code)
write_file(f"{MODEL_NAME}ListView.swift", list_code)
display(Markdown(f"```swift\n{list_code}\n```"))

In [ ]:
# Generate ContentView
if NAV_STYLE == "tabs" and TABS:
    tabs_str = ', '.join(TABS)
    content_code = ask(f"""Write SwiftUI ContentView with TabView.
Tabs: {tabs_str}. First tab shows {MODEL_NAME}ListView.
Reply with ONLY the Swift code.""")
else:
    content_code = ask(f"""Write SwiftUI ContentView that shows {MODEL_NAME}ListView.
Initialize {MODEL_NAME}ViewModel and pass to environment.
Reply with ONLY the Swift code.""")
content_code = extract_code(content_code)
write_file("ContentView.swift", content_code)
display(Markdown(f"```swift\n{content_code}\n```"))

In [ ]:
# Generate App Entry
app_code = ask(f"""Write the @main App struct for {APP_NAME}App.
Use WindowGroup with ContentView. Reply with ONLY the Swift code.""")
app_code = extract_code(app_code)
write_file(f"{APP_NAME}App.swift", app_code)
display(Markdown(f"```swift\n{app_code}\n```"))

---
# Phase 5: Build

In [ ]:
print("🔨 Building...")
result = bash(f'xcodebuild -project "{XCODEPROJ}" -scheme "{APP_NAME}" -destination "generic/platform=iOS" build 2>&1 | tail -20')
if "BUILD SUCCEEDED" in result:
    print("✅ BUILD SUCCEEDED")
else:
    print(result)
    print("\n⚠️ Build failed - check errors above")